# Strategic Prediction with SAP RPT-1: JavaScript SDK (Deno)

This notebook is the **JavaScript SDK counterpart** to the Python SDK tutorial. It achieves the same prediction task — regression (`Days Late`) and classification (`Risk Score`) on a payment transactions dataset — using the **`@sap-ai-sdk/rpt`** package and **Deno** as the runtime.

## What You Will Learn
- How to authenticate with SAP AI Core using environment variables in a Deno notebook
- How to load and parse an Excel file in TypeScript/Deno
- How to split context rows from prediction rows
- How to call SAP RPT-1 using `predictWithoutSchema()` — the primary method (1-to-1 with Python flow)
- **JS-specific enhancements:** `predictWithSchema()` and `predictParquet()` as advanced alternatives

## Structure
| Section | Method | Notes |
|---|---|---|
| Steps 1–6 | `predictWithoutSchema()` | Core flow — mirrors the Python notebook exactly |
| Step 7 (Advanced) | `predictWithSchema()` | JS enhancement — explicit schema for better type inference |
| Step 8 (Advanced) | `predictParquet()` | JS enhancement — file-based prediction |

---

## Prerequisites

- Deno installed ([deno.land](https://deno.land))
- Deno Jupyter kernel configured (`deno jupyter --install`)
- A running **SAP RPT-1 deployment** (see the tutorial's AI Launchpad or Python SDK sections for deployment steps)
- Your SAP AI Core service key saved as a `.env` file (see Step 1 below)
- The dataset file `payment-delay-data.xlsx` in the same directory as this notebook

---
## Step 1: Load Credentials from Environment Variables

In [ ]:
import dotenv from 'npm:dotenv';
dotenv.config();

const serviceKey = JSON.parse(process.env.AICORE_SERVICE_KEY);

console.log('✅ Service key loaded');
console.log('AI API URL:', serviceKey.serviceurls.AI_API_URL);
console.log('Resource Group:', serviceKey.resourcegroup);

---
## Step 2: Install Dependencies & Initialise the RPT Client



In [14]:
import { RptClient } from 'npm:@sap-ai-sdk/rpt';

const serviceKey = JSON.parse(process.env.AICORE_SERVICE_KEY);

const client = new RptClient({
  modelName: 'sap-rpt-1-small',
  resourceGroup: serviceKey.resourcegroup
});

console.log('✅ RPT client initialized');
console.log('Resource Group:', serviceKey.resourcegroup);

✅ RPT client initialized
Resource Group: grounding


### creating deployment

In [17]:
import { ConfigurationApi } from '@sap-ai-sdk/ai-api';

const RESOURCE_GROUP = serviceKey.resourcegroup; // Please change to your desired resource group

// Create orchestration configuration using ConfigurationApi
async function createOrchestrationConfiguration() {
  try {
    const response = await ConfigurationApi
      .configurationCreate({
        name: 'sap-rpt-1-small', // Choose a meaningful name
        executableId: 'aicore-sap', // Orchestration executable ID
        scenarioId: 'foundation-models', // Orchestration scenario ID
      }, {'AI-Resource-Group': RESOURCE_GROUP}).execute();

      return response;
  } catch (error: any) {
      // Handle API errors
      console.error('Configuration creation failed:', error.stack);
  }
}

const configuration = await createOrchestrationConfiguration();
console.log(configuration?.message); // Print the configuration response message

Configuration created


### deployment creation

In [18]:
import { DeploymentApi } from '@sap-ai-sdk/ai-api'; 
import type { AiDeploymentCreationResponse } from '@sap-ai-sdk/ai-api'; 

// Create Orchestration deployment using DeploymentApi
async function createOrchestrationDeployment() { 
 // Extract the configuration ID from the result of the previous step 
  const configurationId = configuration.id;

 try { 
 const response = await DeploymentApi
      .deploymentCreate(
        { configurationId }, 
        { 'AI-Resource-Group': RESOURCE_GROUP }
      ).execute();   

    return response;
 } catch (error: any) { 
    console.error('Deployment creation failed:', error.stack);
} 
} 
 
const deployment = await createOrchestrationDeployment();
console.log(deployment?.message) // Print the deployment creation response

Deployment scheduled.


---
## Step 3: Load and Prepare the Dataset

The dataset is the same Excel file used in the Python notebook — `payment-delay-data.xlsx`.
We use the `xlsx` npm package to read it in Deno.

**Column reference (same as Python tutorial):**

| Column | Role |
|---|---|
| `Transaction ID` | Index column — unique row identifier |
| `Customer ID`, `Customer Type`, `Industry`, `Region`, ... | Feature columns |
| **`Days Late`** | **Regression target** — numeric prediction |
| **`Risk Score`** | **Classification target** — category label + confidence |

In [3]:
import * as XLSX from 'npm:xlsx';

// Configuration — mirrors the Python notebook constants exactly
const INDEX_COLUMN  = 'Transaction ID';
const TARGET_REG    = 'Days Late';     // regression target — numeric prediction
const TARGET_CLS    = 'Risk Score';    // classification target — category + confidence
const PREDICT_TOKEN = '[PREDICT]';

// Load the Excel file
const fileBuffer = await Deno.readFile('payment-delay-data.xlsx');
const workbook   = XLSX.read(fileBuffer, { type: 'buffer' });

// Read the first sheet as an array of row objects
// All values are read as strings to preserve [PREDICT] tokens (matches Python dtype=str)
const sheetName = workbook.SheetNames[0];
const worksheet = workbook.Sheets[sheetName];
const allRows   = XLSX.utils.sheet_to_json<Record<string, string>>(worksheet, {
  raw: false,   // all values as strings — equivalent to pandas dtype=str
  defval: ''    // fill empty cells with '' — equivalent to pandas .fillna('')
});

console.log(`✅ Dataset loaded from sheet: "${sheetName}"`);
console.log(`   Total rows   : ${allRows.length}`);
console.log(`   Columns      : ${Object.keys(allRows[0]).join(', ')}`);
console.log('\nSample row (first record):');
console.log(JSON.stringify(allRows[0], null, 2));

✅ Dataset loaded from sheet: "payment-delay-data"
   Total rows   : 500
   Columns      : Transaction ID, Customer ID, Days Late, Customer Type, Industry, Region, Currency, Invoice Amount, Due Days, Credit Limit, Outstanding, Credit Usage, Relationship Years, Previous Delays, Avg Days Late, Payment Method, Economic Indicator, Quarter, Contact Attempts, Has Dispute, Dispute Amount, Risk Score

Sample row (first record):
{
  "Transaction ID": "TXN AZT67X0T",
  "Customer ID": "CUST 1014",
  "Days Late": "[PREDICT]",
  "Customer Type": "Small Business",
  "Industry": "Education",
  "Region": "Middle East",
  "Currency": "USD",
  "Invoice Amount": "30214.02",
  "Due Days": "15",
  "Credit Limit": "118859.72",
  "Outstanding": "42142.81",
  "Credit Usage": "35.5",
  "Relationship Years": "5",
  "Previous Delays": "5",
  "Avg Days Late": "22.7",
  "Payment Method": "Credit Card",
  "Economic Indicator": "1.1",
  "Quarter": "Q4",
  "Contact Attempts": "1",
  "Has Dispute": "FALSE",
  "Dispute 

---
## Step 4: Separate Context Rows from Prediction Rows

The RPT-1 model uses **in-context learning**:
- **Context rows** — rows with known values in both target columns. The model learns the data pattern from these.
- **Prediction rows** — rows containing `[PREDICT]` in one or both target columns. The model fills these in.

> **Order matters:** Context rows **must** come before prediction rows in the final array sent to the API.

> **How many context rows?** SAP experiments show 1,000–2,000 rows are sufficient to outperform narrow AI models trained on millions of rows.

In [4]:
// Context rows: both targets have known (non-PREDICT, non-empty) values
const contextRows = allRows.filter(row =>
  row[TARGET_REG] !== PREDICT_TOKEN && row[TARGET_REG] !== '' &&
  row[TARGET_CLS] !== PREDICT_TOKEN && row[TARGET_CLS] !== ''
);

// Prediction rows: either target contains [PREDICT]
const predictRows = allRows.filter(row =>
  row[TARGET_REG] === PREDICT_TOKEN ||
  row[TARGET_CLS] === PREDICT_TOKEN
);

console.log(`✅ Row split complete:`);
console.log(`   Context rows  (model learns from these) : ${contextRows.length}`);
console.log(`   Prediction rows (model fills these in)  : ${predictRows.length}`);

// Preview a prediction row
console.log('\nSample prediction row:');
console.log(JSON.stringify(predictRows[0], null, 2));

✅ Row split complete:
   Context rows  (model learns from these) : 469
   Prediction rows (model fills these in)  : 31

Sample prediction row:
{
  "Transaction ID": "TXN AZT67X0T",
  "Customer ID": "CUST 1014",
  "Days Late": "[PREDICT]",
  "Customer Type": "Small Business",
  "Industry": "Education",
  "Region": "Middle East",
  "Currency": "USD",
  "Invoice Amount": "30214.02",
  "Due Days": "15",
  "Credit Limit": "118859.72",
  "Outstanding": "42142.81",
  "Credit Usage": "35.5",
  "Relationship Years": "5",
  "Previous Delays": "5",
  "Avg Days Late": "22.7",
  "Payment Method": "Credit Card",
  "Economic Indicator": "1.1",
  "Quarter": "Q4",
  "Contact Attempts": "1",
  "Has Dispute": "FALSE",
  "Dispute Amount": "0",
  "Risk Score": "High"
}


---
## Step 5: Build the Rows Array

Concatenate context rows first, then prediction rows — this ordering is required by the RPT-1 API.
This directly mirrors the Python line: `all_rows = context_df.to_dict(orient='records') + predict_df.to_dict(orient='records')`

In [5]:
// Context rows FIRST, prediction rows SECOND — required ordering
const rows = [...contextRows, ...predictRows];

console.log(`✅ Combined rows array built:`);
console.log(`   Total rows  : ${rows.length}`);
console.log(`   First row   : context (${rows[0][INDEX_COLUMN]})`);
console.log(`   Last row    : prediction (${rows[rows.length - 1][INDEX_COLUMN]})`);

✅ Combined rows array built:
   Total rows  : 500
   First row   : context (TXN SY4HFYMZ)
   Last row    : prediction (TXN 24H6IYO6)


---
## Step 6: Call the Model with `predictWithoutSchema()` — Core Method

This is the **primary method** and maps 1-to-1 with the Python SDK flow.
`predictWithoutSchema()` sends the rows to the model and lets RPT-1 infer column data types automatically.

Both targets — `Days Late` (regression) and `Risk Score` (classification) — are predicted in a **single API call**.

> **Note:** Ensure your RPT-1 deployment is in **Running** status in AI Launchpad before executing this cell.

In [6]:
console.log(`Sending ${predictRows.length} prediction rows with ${contextRows.length} context rows...`);
console.log('Calling RPT-1 model...');

const response = await client.predictWithoutSchema({
  prediction_config: {
    target_columns: [
      {
        name: TARGET_REG,
        task_type: 'regression',
        prediction_placeholder: PREDICT_TOKEN
      },
      {
        name: TARGET_CLS,
        task_type: 'classification',
        prediction_placeholder: PREDICT_TOKEN
      }
    ]
  },
  index_column: INDEX_COLUMN,
  rows
});

console.log(`\n✅ Predictions received: ${response.predictions.length}`);
console.log('Metadata:', JSON.stringify(response.metadata, null, 2));

Sending 31 prediction rows with 469 context rows...
Calling RPT-1 model...

✅ Predictions received: 31
Metadata: {
  "num_columns": 22,
  "num_predictions": 62,
  "num_query_rows": 31,
  "num_rows": 500
}


### Parse and Display the Results

Extract the prediction value and confidence from the response, then display as a table.

> **Reading the response:**
> - **Regression** — `.prediction` returns the numeric value directly. No `confidence` score for regression.
> - **Classification** — `.prediction` returns the category label. `.confidence` returns a 0–1 score.

In [7]:
// Parse predictions into a flat results array
const results = response.predictions.map(pred => ({
  [INDEX_COLUMN]           : pred[INDEX_COLUMN],
  'Predicted Days Late'    : pred[TARGET_REG]?.[0]?.prediction ?? null,
  'Predicted Risk Score'   : pred[TARGET_CLS]?.[0]?.prediction ?? null,
  'Risk Score Confidence'  : pred[TARGET_CLS]?.[0]?.confidence != null
    ? Number(pred[TARGET_CLS][0].confidence).toFixed(4)
    : null
}));

// Display as a table in the notebook
console.log('\n=== RPT-1 Prediction Results ===\n');
console.table(results);

// Also print a text table for easy reading
console.log('\nDetailed output:');
results.forEach(r => {
  console.log(
    `${r[INDEX_COLUMN].padEnd(20)} | ` +
    `Days Late: ${String(r['Predicted Days Late']).padEnd(10)} | ` +
    `Risk Score: ${String(r['Predicted Risk Score']).padEnd(10)} | ` +
    `Confidence: ${r['Risk Score Confidence']}`
  );
});


=== RPT-1 Prediction Results ===

┌───────┬────────────────┬─────────────────────┬──────────────────────┬───────────────────────┐
│ (idx) │ Transaction ID │ Predicted Days Late │ Predicted Risk Score │ Risk Score Confidence │
├───────┼────────────────┼─────────────────────┼──────────────────────┼───────────────────────┤
│     0 │ "TXN AZT67X0T" │ 20.224315643310547  │ "High"               │ "0.7200"              │
│     1 │ "TXN RWO7A9Z2" │ 19.924007415771484  │ "High"               │ "0.6900"              │
│     2 │ "TXN C18O8PW7" │ 18.13738250732422   │ "High"               │ "0.7500"              │
│     3 │ "TXN MQ5R05JP" │ 41.70020294189453   │ "Medium"             │ "0.6100"              │
│     4 │ "TXN RO3WUTF4" │ 10.779764175415039  │ "High"               │ "0.7800"              │
│     5 │ "TXN 19TQX3BG" │ 21.431804656982422  │ "High"               │ "0.6300"              │
│     6 │ "TXN 6Y9PMTYB" │ 26.455337524414062  │ "Medium"             │ "0.5900"              │
│    

### Export Results to CSV

Save the parsed results to a CSV file — matching the Python notebook's final export step.

In [8]:
// Export results to CSV using the xlsx package's CSV utilities
const exportSheet = XLSX.utils.json_to_sheet(results);
const csvOutput   = XLSX.utils.sheet_to_csv(exportSheet);

await Deno.writeTextFile('rpt1_predictions.csv', csvOutput);

console.log(`✅ Results exported to rpt1_predictions.csv (${results.length} rows)`);
console.log('\nCSV preview (first 5 lines):');
console.log(csvOutput.split('\n').slice(0, 6).join('\n'));

✅ Results exported to rpt1_predictions.csv (31 rows)

CSV preview (first 5 lines):
Transaction ID,Predicted Days Late,Predicted Risk Score,Risk Score Confidence
TXN AZT67X0T,20.22431564,High,0.7200
TXN RWO7A9Z2,19.92400742,High,0.6900
TXN C18O8PW7,18.13738251,High,0.7500
TXN MQ5R05JP,41.70020294,Medium,0.6100
TXN RO3WUTF4,10.77976418,High,0.7800


---
---
## Advanced: Step 7 — `predictWithSchema()` (JS-Specific Enhancement)

This section goes beyond the core Python flow. `predictWithSchema()` lets you **explicitly define the data schema** (column names and data types), which gives the model better semantic understanding of your data structure.

**When to use this instead of `predictWithoutSchema()`:**
- When you know the exact data types of your columns
- When you have numeric-looking identifier columns that should be treated as strings (e.g., postal codes, customer IDs)
- For production workloads where maximum accuracy matters

**Supported `dtype` values:** `'string'` | `'numeric'` | `'date'`

> **Best practice:** Use `dtype: 'string'` for identifier columns, category codes, and postal codes — even when values look numeric. This prevents the model from interpolating them as continuous numbers.

In [9]:
import type { PredictionData } from 'npm:@sap-ai-sdk/rpt';

// Define the explicit schema for the payment transactions dataset
// Use 'as const' to enable TypeScript type inference for PredictionData<typeof schema>
const schema = [
  { name: 'Transaction ID',      dtype: 'string'  },
  { name: 'Customer ID',         dtype: 'string'  },  // identifier — keep as string
  { name: 'Customer Type',       dtype: 'string'  },
  { name: 'Industry',            dtype: 'string'  },
  { name: 'Region',              dtype: 'string'  },
  { name: 'Currency',            dtype: 'string'  },
  { name: 'Invoice Amount',      dtype: 'numeric' },
  { name: 'Due Days',            dtype: 'numeric' },
  { name: 'Credit Limit',        dtype: 'numeric' },
  { name: 'Outstanding',         dtype: 'numeric' },
  { name: 'Credit Usage',        dtype: 'numeric' },
  { name: 'Relationship Years',  dtype: 'numeric' },
  { name: 'Previous Delays',     dtype: 'numeric' },
  { name: 'Avg Days Late',       dtype: 'numeric' },
  { name: 'Payment Method',      dtype: 'string'  },
  { name: 'Economic Indicator',  dtype: 'numeric' },
  { name: 'Quarter',             dtype: 'string'  },
  { name: 'Contact Attempts',    dtype: 'numeric' },
  { name: 'Has Dispute',         dtype: 'string'  },
  { name: 'Dispute Amount',      dtype: 'numeric' },
  { name: 'Days Late',           dtype: 'string'  },  // target — string to preserve [PREDICT]
  { name: 'Risk Score',          dtype: 'string'  }   // target — string to preserve [PREDICT]
] as const;

// Typed prediction data — TypeScript infers valid column names from schema
const predictionData: PredictionData<typeof schema> = {
  prediction_config: {
    target_columns: [
      {
        name: TARGET_REG,
        task_type: 'regression',
        prediction_placeholder: PREDICT_TOKEN
      },
      {
        name: TARGET_CLS,
        task_type: 'classification',
        prediction_placeholder: PREDICT_TOKEN
      }
    ]
  },
  index_column: INDEX_COLUMN,
  rows   // same combined rows array from Step 5
};

console.log('Calling RPT-1 with explicit schema...');

const schemaResponse = await client.predictWithSchema(schema, predictionData);

console.log(`\n✅ Predictions received (with schema): ${schemaResponse.predictions.length}`);

// Parse and display results
const schemaResults = schemaResponse.predictions.map(pred => ({
  [INDEX_COLUMN]           : pred[INDEX_COLUMN],
  'Predicted Days Late'    : pred[TARGET_REG]?.[0]?.prediction ?? null,
  'Predicted Risk Score'   : pred[TARGET_CLS]?.[0]?.prediction ?? null,
  'Risk Score Confidence'  : pred[TARGET_CLS]?.[0]?.confidence != null
    ? Number(pred[TARGET_CLS][0].confidence).toFixed(4)
    : null
}));

console.log('\n=== Results (predictWithSchema) ===\n');
console.table(schemaResults);

Calling RPT-1 with explicit schema...

✅ Predictions received (with schema): 31

=== Results (predictWithSchema) ===

┌───────┬────────────────┬─────────────────────┬──────────────────────┬───────────────────────┐
│ (idx) │ Transaction ID │ Predicted Days Late │ Predicted Risk Score │ Risk Score Confidence │
├───────┼────────────────┼─────────────────────┼──────────────────────┼───────────────────────┤
│     0 │ "TXN AZT67X0T" │ 20.224315643310547  │ "High"               │ "0.7200"              │
│     1 │ "TXN RWO7A9Z2" │ 19.924007415771484  │ "High"               │ "0.6900"              │
│     2 │ "TXN C18O8PW7" │ 18.13738250732422   │ "High"               │ "0.7500"              │
│     3 │ "TXN MQ5R05JP" │ 41.70020294189453   │ "Medium"             │ "0.6100"              │
│     4 │ "TXN RO3WUTF4" │ 10.779764175415039  │ "High"               │ "0.7800"              │
│     5 │ "TXN 19TQX3BG" │ 21.431804656982422  │ "High"               │ "0.6300"              │
│     6 │ "TXN 6Y9

#### Optional: Enable Request Compression

For large datasets, you can enable gzip compression to improve performance.
By default, requests ≥ 1024 bytes are compressed automatically (`mode: 'auto'`).
The example below forces compression and adds resilience middleware.

In [10]:
import { resilience } from 'npm:@sap-cloud-sdk/resilience';

// predictWithSchema with compression + resilience (circuit breaker + timeout)
const resilientResponse = await client.predictWithSchema(schema, predictionData, {
  compress: {
    mode: 'always'           // force compression regardless of payload size
  },
  middleware: resilience({
    timeout: 60000,          // 60-second timeout for large payloads
    circuitBreaker: true,    // circuit breaker enabled by default
    retry: 2                 // retry up to 2 times on transient failures
  })
});

console.log(`✅ Resilient prediction complete: ${resilientResponse.predictions.length} results`);

✅ Resilient prediction complete: 31 results


---
## Advanced: Step 8 — `predictParquet()` (JS-Specific Enhancement)

The `/predict-parquet` endpoint accepts a **Parquet file** instead of a JSON rows array.

### 📌 Shared Prerequisite: `payment_transactions.parquet`

The Parquet file is **created once** in the Python SDK notebook and **reused as-is** across all three approaches:

| Approach | Parquet file role |
|---|---|
| **Python SDK** | Creates `payment_transactions.parquet` via `df.to_parquet()` |
| **JavaScript SDK** | Reads the same file from disk and sends it via `predictParquet()` |
| **Bruno / REST API** | Attaches the same file as a `multipart/form-data` upload |
| **AI Launchpad** | Uploads the same file through the UI |

> **Before running Step 8b below**, make sure you have already run the Python SDK notebook's parquet export cell. The file `payment_transactions.parquet` should be present in the same directory as this notebook.

This is exactly how the tutorial is structured across all three approaches — the Python SDK owns the data preparation (Excel → Parquet), and the other approaches simply consume that output.

### Step 8b: Send the Shared Parquet File to RPT-1

Read `payment_transactions.parquet` from disk and send it via `predictParquet()`.
The prediction config is identical to Steps 6 and 7 — same targets, same index column.

> **`parse_data_types: true`** (the default) tells RPT-1 to infer numeric and date columns from the string values in the Parquet file — equivalent to how the Python SDK handles type inference automatically.

In [11]:
// Read payment_transactions.parquet — created once by the Python SDK notebook.
// The same file is reused in Bruno and AI Launchpad too — no re-creation needed.
const parquetPath = 'payment_transactions.parquet';

// Verify the file exists before sending
try {
  const fileStat = await Deno.stat(parquetPath);
  console.log(`Found ${parquetPath} (${fileStat.size} bytes) — ready to send.`);
} catch {
  throw new Error(
    `${parquetPath} not found. Run the Python SDK notebook parquet export cell first.`
  );
}

const parquetBuffer = await Deno.readFile(parquetPath);
const parquetBlob   = new Blob([parquetBuffer], {
  type: 'application/vnd.apache.parquet'
});

console.log(`Sending ${parquetPath} (${parquetBuffer.byteLength} bytes) to RPT-1...`);

const parquetResponse = await client.predictParquet({
  file: parquetBlob,
  prediction_config: {
    target_columns: [
      { name: TARGET_REG, task_type: 'regression',     prediction_placeholder: PREDICT_TOKEN },
      { name: TARGET_CLS, task_type: 'classification', prediction_placeholder: PREDICT_TOKEN }
    ]
  },
  index_column: INDEX_COLUMN
  // parse_data_types defaults to true — RPT-1 infers numeric/date types from string values
});

console.log(`\n✅ Parquet predictions received: ${parquetResponse.predictions.length}`);
console.log('Metadata:', JSON.stringify(parquetResponse.metadata, null, 2));

const parquetResults = parquetResponse.predictions.map(pred => ({
  [INDEX_COLUMN]          : pred[INDEX_COLUMN],
  'Predicted Days Late'   : pred[TARGET_REG]?.[0]?.prediction ?? null,
  'Predicted Risk Score'  : pred[TARGET_CLS]?.[0]?.prediction ?? null,
  'Risk Score Confidence' : pred[TARGET_CLS]?.[0]?.confidence != null
    ? Number(pred[TARGET_CLS][0].confidence).toFixed(4)
    : null
}));

console.log('\n=== Results (predictParquet) ===\n');
console.table(parquetResults);


Found payment_transactions.parquet (36182 bytes) — ready to send.
Sending payment_transactions.parquet (36182 bytes) to RPT-1...

✅ Parquet predictions received: 31
Metadata: {
  "num_columns": 22,
  "num_predictions": 62,
  "num_query_rows": 31,
  "num_rows": 500
}

=== Results (predictParquet) ===

┌───────┬────────────────┬─────────────────────┬──────────────────────┬───────────────────────┐
│ (idx) │ Transaction ID │ Predicted Days Late │ Predicted Risk Score │ Risk Score Confidence │
├───────┼────────────────┼─────────────────────┼──────────────────────┼───────────────────────┤
│     0 │ "TXN AZT67X0T" │ 20.224315643310547  │ "High"               │ "0.7200"              │
│     1 │ "TXN RWO7A9Z2" │ 19.924007415771484  │ "High"               │ "0.6900"              │
│     2 │ "TXN C18O8PW7" │ 18.13738250732422   │ "High"               │ "0.7500"              │
│     3 │ "TXN MQ5R05JP" │ 41.70020294189453   │ "Medium"             │ "0.6100"              │
│     4 │ "TXN RO3WUTF4" │

---
## Summary

You have now completed the SAP RPT-1 tutorial using the **JavaScript SDK (Deno)**.

| What You Did | Method | JS Notebook Step |
|---|---|---|
| Loaded credentials from `.env` | `npm:dotenv` + `process.env` | Step 1 |
| Initialised the RPT client with deployment ID | `new RptClient({ deploymentId, resourceGroup })` | Step 2 |
| Loaded Excel dataset | `xlsx` npm package | Step 3 |
| Split context / prediction rows | Array filter | Step 4 |
| Built combined rows array | Spread concat | Step 5 |
| Called the model (core flow) | `predictWithoutSchema()` | Step 6 |
| Parsed & exported results | `console.table`, CSV | Step 6 |
| Called with explicit schema | `predictWithSchema()` | Step 7 (Advanced) |
| Sent Parquet file | `predictParquet()` | Step 8 (Advanced) |

## Next Steps

- To use `sap-rpt-1-large`: deploy it in AI Launchpad, copy its deployment ID, update `AICORE_DEPLOYMENT_ID` in `.env`
- Add more target columns — RPT-1 supports up to 10 simultaneously
- Add resilience middleware (`@sap-cloud-sdk/resilience`) for production workloads
- Explore the [SAP AI SDK JS documentation](https://sap.github.io/ai-sdk/docs/js/rpt) for full API reference

## Useful References

- [SAP RPT-1 JS SDK Docs](https://sap.github.io/ai-sdk/docs/js/rpt)
- [SAP Help Portal — SAP RPT-1](https://help.sap.com/docs/sap-ai-core/generative-ai/sap-rpt-1)
- [SAP AI SDK npm packages](https://www.npmjs.com/org/sap-ai-sdk)
- [SAP-samples/sap-rpt-samples on GitHub](https://github.com/SAP-samples/sap-rpt-samples)